<a href="https://colab.research.google.com/github/FilipeSchweitzer/ProjetoAplicado2_CDIA_UniSenai/blob/main/IST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd

df_ident = pd.read_excel('IDENTIFICACAO.xlsx', usecols=['Compound', 'Compound ID', 'Formula', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description'])
df_abund = pd.read_excel('ABUND.xlsx', usecols=['Compound', 'Identifications'])

df = pd.merge(df_ident, df_abund, on='Compound')

csv = df.to_csv('table.csv', index=False)

df = pd.read_csv('table.csv')

In [75]:
import requests
import json

def Pubchem_search(name:str):
    result = {}

    main = Pubchem_main_search(name)
    #taxonomy = Pubchem_taxonomy(name)

    for key, value in main.items():
        result[key] = value

    return main

def Pubchem_main_search(name:str):
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json()

        if data != 404:
            props = data['PC_Compounds'][0]['props']
            return {p['urn']['label']: p['value'] for p in props
                    if p['urn']['label'] in ['IUPAC Name', 'SMILES']}

def Pubchem_taxonomy(name:str):
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/description/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json()
        return data
    elif code != 404:
        return f'Error: {code}'

# print(Pubchem_search('Adozelesin'))

In [27]:
tabela_final = pd.DataFrame(columns=['Compound ID', 'Description', 'IUPAC Name', 'Formula', 'SMILES', 'Fragmentation Score', 'Score', 'Isotope Similarity',
                        'Identifications'])

def add_to_final(index, info):
    original_table = df.iloc[index].fillna('Nan')
    api_search = info

    # print(original_table)

    data = []
    for column in tabela_final.columns:
        if column in original_table.index:
            #print(f'{column}: {original_table.loc[column]}')
            data.append(original_table[column])
        else:
            if info is str:
                data.append(info)
            elif column in info.keys():
                #print(f'{column}: {info[column]['sval']}')
                data.append(info[column]['sval'])
            else:
                data.append('Nan')

    print(data)

    # adiciona os valores a tabela final
    if len(data) == len(tabela_final.columns):
        tabela_final.loc[len(tabela_final)] = data
    else:
        print(f'\nHÁ COLUNAS EXTRAS !!!\n {data}\n{tabela_final.columns}')

In [28]:
#exvazia a tabela
def clear_table():
    import os
    global tabela_final
    tabela_final = tabela_final.iloc[0:0]

    arquivo = 'final.csv'
    if os.path.exists(arquivo):
        os.remove(arquivo)

clear_table()

MAIN

In [76]:
clear_table()

from time import sleep

found = 0
for index in range(50, 60):
    try:
        compound = df.iloc[index]['Description']

        search = Pubchem_search(compound)

        if search != 'Error: 404':
            found += 1
            print(f'{index}.{compound} / found:{found}')
            print(search)
            #add_to_final(index, search)
    except Exception as e:
        print(f"\nERRO!!! índice:{index}, erro:{e}\n")
        continue
    sleep(0.2)

#tabela_final.to_excel('final.xlsx', index=False)
tabela_final.to_csv('final.csv', index=False)
print('finished')

50.nan / found:1
{'IUPAC Name': {'sval': '(2S,4S,5R,6R)-5-acetamido-2,4-dihydroxy-6-[(1R,2R)-1,2,3-trihydroxypropyl]tetrahydropyran-2-carboxylic acid'}, 'SMILES': {'sval': 'CC(=O)NC1C(CC(OC1C(C(CO)O)O)(C(=O)O)O)O'}}
51.Adozelesin / found:2
{'IUPAC Name': {'sval': 'N-[2-[(1R,12S)-7-keto-3-methyl-5,10-diazatetracyclo[7.4.0.01,12.02,6]trideca-2(6),3,8-triene-10-carbonyl]-1H-indol-5-yl]coumarilamide'}, 'SMILES': {'sval': 'CC1=CNC2=C1C34CC3CN(C4=CC2=O)C(=O)C5=CC6=C(N5)C=CC(=C6)NC(=O)C7=CC8=CC=CC=C8O7'}}

ERRO!!! índice:52, erro:'NoneType' object has no attribute 'items'


ERRO!!! índice:53, erro:'NoneType' object has no attribute 'items'

54.Adozelesin / found:3
{'IUPAC Name': {'sval': 'N-[2-[(1R,12S)-7-keto-3-methyl-5,10-diazatetracyclo[7.4.0.01,12.02,6]trideca-2(6),3,8-triene-10-carbonyl]-1H-indol-5-yl]coumarilamide'}, 'SMILES': {'sval': 'CC1=CNC2=C1C34CC3CN(C4=CC2=O)C(=O)C5=CC6=C(N5)C=CC(=C6)NC(=O)C7=CC8=CC=CC=C8O7'}}
55.Adozelesin / found:4
{'IUPAC Name': {'sval': 'N-[2-[(1R,12S)-7-keto